# 05 — Model Interpretation (SHAP)

**Goal**: Understand *why* the model flags transactions as fraud  
**Method**: SHAP TreeExplainer (TreeSHAP — fast for gradient boosting)

### Fraud SHAP Analysis Structure
1. Global feature importance (what matters most overall)
2. Beeswarm — direction of feature effects
3. Waterfall — explain individual fraud transaction
4. Dependence plots — key feature interactions
5. Error analysis — false positives vs false negatives profile

---

In [ ]:
import sys; sys.path.insert(0, '..')
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

from src.data.loader import load_train_test
from src.data.features import engineer_features, split_features_target
from src.models.evaluation import compute_fraud_metrics
from src.pipeline import load_config

cfg = load_config('../configs/config.yaml')
BEST_MODEL = '../outputs/models/xgboost_model.joblib'  # Update to your best

artifact  = joblib.load(BEST_MODEL)
pipeline  = artifact['pipeline']
threshold = artifact['threshold']
print(f'Loaded model | threshold={threshold:.3f}')

In [ ]:
_, df_test_raw = load_train_test('../data/fraudTrain.csv', '../data/fraudTest.csv')
df_test = engineer_features(df_test_raw)
num_feats = cfg['features']['numeric']
cat_feats = cfg['features']['categorical']
X_test, y_test = split_features_target(df_test, feature_names=num_feats+cat_feats)

preprocessor = pipeline.named_steps['preprocessor']
classifier   = pipeline.named_steps['classifier']
X_transformed = preprocessor.transform(X_test)

try:
    ohe_names = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(cat_feats).tolist()
except:
    ohe_names = [f'cat_{i}' for i in range(X_transformed.shape[1] - len(num_feats))]

feat_names = num_feats + ohe_names
X_df = pd.DataFrame(X_transformed, columns=feat_names)

print(f'Test: {X_test.shape} | Transformed: {X_transformed.shape}')

## 1. SHAP Values Computation

In [ ]:
# Sample for speed on large test set
np.random.seed(42)
n_sample  = min(5000, len(X_df))
idx       = np.random.choice(len(X_df), n_sample, replace=False)
X_sample  = X_df.iloc[idx]
y_sample  = y_test.values[idx]

explainer   = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_sample)

# For binary classification models
if isinstance(shap_values, list):
    sv = shap_values[1]  # Class 1 = Fraud
else:
    sv = shap_values

y_prob_sample = classifier.predict_proba(X_sample)[:, 1]
print(f'SHAP values: {sv.shape} | Fraud in sample: {y_sample.sum():,} ({y_sample.mean():.2%})')

## 2. Global Feature Importance

In [ ]:
mean_abs_shap = np.abs(sv).mean(axis=0)
imp_df = pd.DataFrame({'Feature': feat_names, 'Mean |SHAP|': mean_abs_shap})
imp_df = imp_df.sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)

print('Top 15 features by |SHAP|:')
print(imp_df.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 7))
top = imp_df.head(15)
ax.barh(top['Feature'][::-1], top['Mean |SHAP|'][::-1],
        color=sns.color_palette('RdYlGn_r', 15), alpha=0.85)
ax.set_xlabel('Mean |SHAP Value| (impact on fraud probability)')
ax.set_title('Global Feature Importance (SHAP)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Beeswarm Plot — Direction of Effects

In [ ]:
shap.summary_plot(sv, X_sample, feature_names=feat_names,
                   max_display=15, show=False, plot_size=(12, 7))
plt.title('SHAP Beeswarm — Feature Impact on Fraud Probability', fontsize=12)
plt.tight_layout(); plt.show()

## 4. Waterfall — Individual Fraud Explanation

In [ ]:
fraud_idx  = np.where((y_prob_sample > 0.7) & (y_sample == 1))[0]
legit_idx  = np.where((y_prob_sample < 0.1) & (y_sample == 0))[0]

base_val = explainer.expected_value if not isinstance(explainer.expected_value, list) \
           else explainer.expected_value[1]

if len(fraud_idx) > 0:
    i = fraud_idx[0]
    print(f'--- Actual Fraud, High Prob={y_prob_sample[i]:.3f} ---')
    exp = shap.Explanation(
        values=sv[i], base_values=base_val,
        data=X_sample.iloc[i].values, feature_names=feat_names
    )
    plt.figure(figsize=(12, 6))
    shap.waterfall_plot(exp, max_display=12, show=False)
    plt.title(f'Why This Transaction Was Flagged (fraud prob={y_prob_sample[i]:.3f})',
              fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
if len(legit_idx) > 0:
    i = legit_idx[0]
    print(f'--- Legitimate Transaction, Low Prob={y_prob_sample[i]:.3f} ---')
    exp = shap.Explanation(
        values=sv[i], base_values=base_val,
        data=X_sample.iloc[i].values, feature_names=feat_names
    )
    plt.figure(figsize=(12, 6))
    shap.waterfall_plot(exp, max_display=12, show=False)
    plt.title(f'Why This Transaction Was NOT Flagged (prob={y_prob_sample[i]:.3f})', fontsize=11)
    plt.tight_layout(); plt.show()

## 5. Dependence Plots — Key Feature Interactions

In [ ]:
top3 = imp_df['Feature'].head(3).tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, feat in zip(axes, top3):
    if feat not in X_sample.columns: ax.set_visible(False); continue
    fi = feat_names.index(feat)
    sc = ax.scatter(
        X_sample[feat].values, sv[:, fi],
        c=y_prob_sample, cmap='RdYlBu_r', alpha=0.3, s=8, vmin=0, vmax=1
    )
    ax.axhline(0, color='grey', ls='--', lw=1)
    ax.set_xlabel(feat); ax.set_ylabel(f'SHAP({feat})')
    ax.set_title(f'SHAP Dependence: {feat}', fontweight='bold')

plt.colorbar(sc, ax=axes[-1], label='Predicted Fraud Prob')
plt.suptitle('Dependence Plots (color = fraud probability)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Error Analysis: FP vs FN Profile

In [ ]:
y_prob_all = pipeline.predict_proba(X_test)[:, 1]
y_pred_all = (y_prob_all >= threshold).astype(int)

err_df = df_test.copy()
err_df['fraud_prob'] = y_prob_all
err_df['predicted']  = y_pred_all
err_df['error'] = 'Correct'
err_df.loc[(y_pred_all==1) & (y_test.values==0), 'error'] = 'FP (False Alarm)'
err_df.loc[(y_pred_all==0) & (y_test.values==1), 'error'] = 'FN (Missed Fraud)'

print('Prediction Breakdown:')
print(err_df['error'].value_counts().to_string())

profile_feats = [f for f in ['log_amt','log_distance_km','age','hour'] if f in err_df.columns]
profile = err_df.groupby('error')[profile_feats].mean().round(3)
print('\nError Type Feature Profiles:')
print(profile.to_string())

## SHAP Insights Summary

| Feature | Direction | Interpretation |
|---------|-----------|----------------|
| `log_distance_km` | ↑ SHAP high values | Far merchant = fraud signal |
| `log_amt` | ↑ SHAP high amounts | High transaction → higher fraud risk |
| `amt_to_pop_ratio` | ↑ SHAP high values | Big spend in small city = suspicious |
| `is_night` | ↑ SHAP when =1 | Night transactions riskier |
| `category_*` | Varies | Some categories inherently riskier |

> **Next**: → `06_Business_Report.ipynb`